# Cube 计算模型

本节从一个具体问题开始：输入两个 `[128, 128]` FP16 矩阵，如何让 Cube 计算它们的乘积，并把 FP32 结果写回 GM？我们先完成单核、单 Tile 的全部流程，再把 K 从 128 扩大到 384。后一种情况中，输出尺寸不变，但同一个输出 Tile 需要累加 3 次。


## 环境准备

运行代码前，请完成 [01.06 环境配置](../01_pypto_pro_introduction/01.06_env_and_first_kernel.ipynb)。

在本节所在目录启动 Notebook，按顺序执行单元即可；不需要先运行其他 Notebook。设备号默认取 `TILE_FWK_DEVICE_ID`，未设置时使用 0。


In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)


: 

## 1. 从矩阵规格到六步流程

设 `A[M,K]`、`B[K,N]`，则 `C[M,N]` 的元素满足：

$$C_{m,n}=\sum_{k=0}^{K-1} A_{m,k} B_{k,n}$$

M、N 决定输出区域，K 决定每个输出元素要累加多少项。本节固定 FP16 输入、FP32 累加与输出，先取 `M=K=N=128`，暂不引入转置、偏置或尾块。

Cube 的操作数来自 L0A/L0B，结果放在 L0C。因此，读完 GM 输入后，还要经过 L1 与 L0 之间的搬运。实现可分为六步：

1. 创建矩阵输入暂存、左操作数、右操作数和结果 Tile。
2. 用 `load` 将 A、B 从 GM 搬到 L1。
3. 用 `move` 将 A 搬到 L0A、B 搬到 L0B。
4. 用 `matmul` 计算，结果写入 L0C。
5. 用 `store` 将结果从 L0C 写回 GM。
6. 为以上操作建立同步依赖，使消费者在数据准备好后执行。

第 6 步约束前面所有操作的时序，并不是写回之后再补一个同步。计算与搬运写在 `pl.section_cube()` 内，TileGroup 的声明可以放在域外。
<img style="display:block; margin-left:0; margin-right:auto;" src="./images/cube_matrix_computation_data_flow.svg" alt="Cube 数据流及 MTE2、MTE1、M、FIX 流水" width="700px">



## 2. 先算清楚 Buffer 地址

一个 FP16 的 `128×128` Tile 占 `128×128×2 = 32768 B = 32 KiB = 0x8000 B`；一个同尺寸 FP32 结果占 `64 KiB = 0x10000 B`。这些形状已满足分形对齐，因而本例可以直接用元素数计算容量。

下面使用单缓冲。地址区间左闭右开，地址单位均为字节：
<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">TileGroup</th><th style="text-align:left;">MemorySpace / Buffer</th><th style="text-align:left;">用途与默认布局</th><th style="text-align:left;">地址区间</th></tr>
<tr><td>a_l1</td><td>Mat / L1</td><td>左输入暂存，NZ</td><td>[0x00000, 0x08000)</td></tr>
<tr><td>b_l1</td><td>Mat / L1</td><td>右输入暂存，NZ</td><td>[0x08000, 0x10000)</td></tr>
<tr><td>a_left</td><td>Left / L0A</td><td>左操作数，NZ</td><td>[0x0000, 0x8000)</td></tr>
<tr><td>b_right</td><td>Right / L0B</td><td>右操作数，ZN</td><td>[0x0000, 0x8000)</td></tr>
<tr><td>acc</td><td>Acc / L0C</td><td>FP32 累加器，NZ</td><td>[0x0000, 0x10000)</td></tr>
</table>


A、B 的 L1 暂存处于同一个地址空间，必须避开彼此。L0A、L0B、L0C 则独立编址，所以三个 `0x0000` 可以同时使用。这里每个 TileGroup 只有一个 mutex ID，对应一个槽位；读取当前槽位用 `current()`。

声明时还有两项容易混淆：`shape` 是 Tile 的物理容量规格，`valid_shape` 是本次参与运算的范围。本例二者都是 `[128,128]`。`fractal=1024` 是 L0C 的 FP32 结果分形大小，指字节数，不是元素数。


## 3. 完整示例：单核单 Tile

下面 5 个 TileGroup 对应上表的 5 段地址。`auto_mutex=True` 结合 `mutex_ids` 建立流水间的读写依赖，便于先聚焦数据路径。`pl.load`、`pl.store` 的 offset 按**逻辑元素坐标**计数；这里所有矩阵都从 `[0,0]` 开始。

阅读时沿着 `cur_a → al` 和 `cur_b → br` 两条路径走到 `ac`，即可把声明、搬运和矩阵乘逐一对应起来。


In [ ]:
@pl.jit(auto_mutex=True)
def mm_single(
    a: pl.Tensor[[128, 128], pl.DT_FP16],
    b: pl.Tensor[[128, 128], pl.DT_FP16],
    out: pl.Tensor[[128, 128], pl.DT_FP32],
):
    a_l1 = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Mat, valid_shape=[128, 128]),
        addrs=0x00000, mutex_ids=[0])
    b_l1 = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Mat, valid_shape=[128, 128]),
        addrs=0x08000, mutex_ids=[1])
    a_left = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Left, valid_shape=[128, 128]),
        addrs=0x0000, mutex_ids=[2])
    b_right = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Right, valid_shape=[128, 128]),
        addrs=0x0000, mutex_ids=[3])
    acc = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP32,
                         target_memory=pl.MemorySpace.Acc, valid_shape=[128, 128],
                         fractal=1024),
        addrs=0x0000, mutex_ids=[4])

    with pl.section_cube():
        cur_a = a_l1.current()
        cur_b = b_l1.current()
        al = a_left.current()
        br = b_right.current()
        ac = acc.current()
        pl.load(cur_a, a, [0, 0])
        pl.load(cur_b, b, [0, 0])
        pl.move(al, cur_a)
        pl.move(br, cur_b)
        pl.matmul(ac, al, br)
        pl.store(out, ac, [0, 0])


### 在 Host 上构造输入并判定结果

单 Tile 示例仅处理一块输出，因此显式用 `mm_single[None, 1]` 启动一个核；`None` 表示使用当前 stream；未切换 stream 时通常为默认 stream。若直接增加核数，各核仍会写同一个输出位置，需要先完成 04.04 的任务切分。

校验用输入实际量化后的 FP16 数据转成 FP32 再做矩阵乘。输出先填 NaN：如果遗漏写回，后面的有限值检查会失败。`synchronize()` 等待设备工作完成，`assert_close` 检查逐元素误差；打印的最大绝对误差只是辅助观察，不能单独替代容差判断。


In [ ]:
torch.manual_seed(42)
a = torch.randn(128, 128, device=device, dtype=torch.float16)
b = torch.randn(128, 128, device=device, dtype=torch.float16)
out = torch.full((128, 128), float('nan'), device=device, dtype=torch.float32)

mm_single[None, 1](a, b, out)
torch.npu.synchronize()
golden = a.float() @ b.float()
assert torch.isfinite(out).all().item(), '存在未写回位置或 NaN/Inf'
torch.testing.assert_close(out, golden, rtol=1e-2, atol=1e-2)
print('单 Tile 校验通过，max_abs =', (out - golden).abs().max().item())


**如何理解结果？** 此处不预填“运行通过”的输出；只有在 950 上实际执行成功才会打印通过信息。可以把两份输入都改为全 1，此时每个结果元素应为 128。随机输入检查一般情形，全 1 输入便于手算，两者互为补充。


## 4. 四条流水线：为什么 Python 顺序还不够

Python 中先写 `load`、再写 `move`，并不等于硬件已经完成前一个动作。它们下发到不同的异步流水，必须建立“数据已就绪”的依赖：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">流水线</th><th style="text-align:left;">含义</th><th style="text-align:left;">典型操作</th></tr>
<tr><td>MTE2</td><td>GM → L1 搬运</td><td>pl.load / pl.load_tile</td></tr>
<tr><td>MTE1</td><td>L1 → L0A/L0B 搬运</td><td>pl.move</td></tr>
<tr><td>M</td><td>矩阵计算</td><td>pl.matmul / pl.matmul_acc</td></tr>
<tr><td>FIX</td><td>L0C → GM 搬运</td><td>pl.store / pl.store_tile</td></tr>
</table>

同步选择与第 2 章结论一致：**make_tile_group + auto_mutex=True 自动插入**（常规场景）；make_tile + 显式 `sync_src/sync_dst` 对（需精确控制时序时）。以下仅展示单次计算的手动同步片段：其中 a_l1/b_l1/a_left/b_right/acc 必须是另用 make_tile 创建的 **Tile**，不是完整示例中的 TileGroup；对应 Kernel 关闭 auto_mutex，不配置 mutex_ids。循环复用缓冲还要补读完→覆写的反向依赖。

```python
with pl.section_cube():
    pl.load(a_l1, a, [0, 0]); pl.load(b_l1, b, [0, 0])
    pl.system.sync_src(set_pipe=pl.PipeType.MTE2, wait_pipe=pl.PipeType.MTE1, event_id=0)
    pl.system.sync_dst(set_pipe=pl.PipeType.MTE2, wait_pipe=pl.PipeType.MTE1, event_id=0)

    pl.move(a_left, a_l1); pl.move(b_right, b_l1)
    pl.system.sync_src(set_pipe=pl.PipeType.MTE1, wait_pipe=pl.PipeType.M, event_id=1)
    pl.system.sync_dst(set_pipe=pl.PipeType.MTE1, wait_pipe=pl.PipeType.M, event_id=1)

    pl.matmul(acc, a_left, b_right)
    pl.system.sync_src(set_pipe=pl.PipeType.M, wait_pipe=pl.PipeType.FIX, event_id=2)
    pl.system.sync_dst(set_pipe=pl.PipeType.M, wait_pipe=pl.PipeType.FIX, event_id=2)
    pl.store(out, acc, [0, 0])
```

这个片段用于阅读同步关系，不作为独立 Kernel 运行。sync_src 由生产流水 SET、sync_dst 由消费流水 WAIT，两者 set_pipe/wait_pipe/event_id 必须一致；静态 event_id 范围为 [0,7]。


再考虑下一轮复用：MTE1 读完某个 L1 槽位后，MTE2 才能覆盖它；M 读完 L0A/L0B 后，MTE1 才能写入新操作数；FIX 读完 L0C 后，下一次计算才可以覆盖结果。只列出“搬入 → 计算 → 搬出”三条正向依赖，仍不足以保证循环复用安全。这也是本章完整示例统一使用 TileGroup 与自动同步的原因。


## 5. K 分块：输出不变，累加次数增加

现在取 `A[128,384]`、`B[384,128]`，每次只搬入 K 方向 128 个元素：

$$C=A[:,0:128]B[0:128,:]+A[:,128:256]B[128:256,:]+A[:,256:384]B[256:384,:]$$

每一项都是 `[128,128]` 的部分结果。首块用 `matmul` 初始化累加器，后两块用 `matmul_acc(ac, ac, al, br)` 读出并更新同一个累加器，最后只写回一次。
<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">k 起点</th><th style="text-align:left;">A / B 搬入坐标</th><th style="text-align:left;">累加器更新</th><th style="text-align:left;">是否写回</th></tr>
<tr><td>0</td><td>[0,0] / [0,0]</td><td>ac = A₀B₀</td><td>否</td></tr>
<tr><td>128</td><td>[0,128] / [128,0]</td><td>ac = ac + A₁B₁</td><td>否</td></tr>
<tr><td>256</td><td>[0,256] / [256,0]</td><td>ac = ac + A₂B₂</td><td>循环结束后写回</td></tr>
</table>


这里把 L1 和 L0A/L0B 改成双缓冲。一个 A 输入槽位为 `0x8000 B`，两个槽位占 `[0,0x10000)`，所以 B 的 L1 起点必须移到 `0x10000`，不能沿用单缓冲的 `0x8000`。L0A、L0B 各预留 `64 KiB`，L0C 仍只需要一个 `64 KiB` 结果槽位。

每轮对输入组调用一次 `next()`，槽位按深度轮转；`ac` 在循环外取得，始终指向同一个累加器。双缓冲给相邻迭代的搬运与计算重叠创造条件，是否获得加速还需在 950 上测量，不能仅凭声明了两个槽位判断。


In [ ]:
@pl.jit(auto_mutex=True)
def mm_k_split(
    a: pl.Tensor[[128, 384], pl.DT_FP16],
    b: pl.Tensor[[384, 128], pl.DT_FP16],
    out: pl.Tensor[[128, 128], pl.DT_FP32],
):
    a_l1 = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Mat, valid_shape=[128, 128]),
        addrs=0x00000, mutex_ids=[0, 1])
    b_l1 = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Mat, valid_shape=[128, 128]),
        addrs=0x10000, mutex_ids=[2, 3])
    a_left = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Left, valid_shape=[128, 128]),
        addrs=0x0000, mutex_ids=[4, 5])
    b_right = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Right, valid_shape=[128, 128]),
        addrs=0x0000, mutex_ids=[6, 7])
    acc = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP32,
                         target_memory=pl.MemorySpace.Acc, valid_shape=[128, 128],
                         fractal=1024),
        addrs=0x0000, mutex_ids=[8])

    with pl.section_cube():
        ac = acc.current()
        for k in pl.range(0, 384, 128):
            cur_a = a_l1.next()
            cur_b = b_l1.next()
            al = a_left.next()
            br = b_right.next()
            pl.load(cur_a, a, [0, k])
            pl.load(cur_b, b, [k, 0])
            pl.move(al, cur_a)
            pl.move(br, cur_b)
            if k == 0:
                pl.matmul(ac, al, br)
            else:
                pl.matmul_acc(ac, ac, al, br)
        pl.store(out, ac, [0, 0])


### 用一个能定位错误的输入检查累加

令 A 全为 1，B 沿 K 方向的三个块分别填 1、2、3，那么每个输出元素都应为 `128×1 + 128×2 + 128×3 = 768`。如果把三个步骤都写成 `matmul`，最终只保留最后一块的 384；若漏掉一个块，也能从差值追查。


In [ ]:
a_k = torch.ones(128, 384, device=device, dtype=torch.float16)
b_k = torch.ones(384, 128, device=device, dtype=torch.float16)
b_k[128:256, :] = 2
b_k[256:384, :] = 3
out_k = torch.full((128, 128), float('nan'), device=device, dtype=torch.float32)

mm_k_split[None, 1](a_k, b_k, out_k)
torch.npu.synchronize()
assert torch.isfinite(out_k).all().item()
torch.testing.assert_close(out_k, a_k.float() @ b_k.float(), rtol=1e-2, atol=1e-2)
torch.testing.assert_close(out_k, torch.full_like(out_k, 768), rtol=0, atol=0)
print('K 分块校验通过：所有元素均为 768')


### 常见现象与排查方向

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">现象</th><th style="text-align:left;">先检查什么</th></tr>
<tr><td>单 Tile 正确，改成双缓冲后出错</td><td>L1 的 B 起点是否仍是 0x8000；两个 A 槽位会与它重叠</td></tr>
<tr><td>K 分块的常量用例得到 384</td><td>是否每轮都用了 matmul，覆盖了前两次结果</td></tr>
<tr><td>第一轮结果异常或多次运行不稳定</td><td>是否首块直接使用 matmul_acc，读取了未初始化的累加器</td></tr>
<tr><td>输出仍包含 NaN</td><td>是否完成 store，写回 offset 和启动核数是否正确</td></tr>
</table>

这些是定位线索，不能代替实际错误信息；随机输入校验将在章节实践中补上。


## 6. 课后练习

1. （填空）Cube 四条流水线按数据流顺序是 MTE2 → ______ → ______ → ______。
2. （选择）`pl.move` 在本节中的用途是：A. GM → L1；B. L1 → L0A/L0B；C. L0C → GM。
3. （推导）L1 的 A、B 都使用两个 `[128,128]` FP16 槽位。若 A 从 `0x00000` 开始，B 最早能从哪里开始？总共预留多少字节？
4. （实践）把 K 分块示例中的三个 B 块改为 1、0、−1。每个结果应是多少？若全部使用 `matmul`，又会得到什么？说明首块初始化和后续累加各自的作用。


In [ ]:
!cat ./answer/04.02_answer.txt


## 7. 本节小结

你已经把矩阵规格落到了五类 Tile 角色、四条流水与完整的 Host 校验。单 Tile 示例的重点是数据路径，K 分块示例新增了输入槽位轮转与累加器生命周期。扩展代码时，地址预算和初始化语义都要随之检查。

下一节 [矩阵分形布局](./04.03_fractal_layout.ipynb) 将解释：同一个逻辑 `[K,N]`，为什么在 L1 与 L0B 会采用不同的元素排列？
